#  March Machine Learning Mania 2026
## Predicting NCAA Men's & Women's Tournament Outcomes
**Goal:** Minimize Brier score across all possible 2026 matchups  
**Approach:** Elo ratings + Team efficiency + Massey consensus rankings → XGBoost

# 1. Data Exploration
Key questions:
- How many seasons of box score data do we have for efficiency features?
- Which Massey ranking systems are available pre-tournament?
- What exactly do we need to predict for 2026?

In [1]:
import pandas as pd
DATA_DIR = '/kaggle/input/competitions/march-machine-learning-mania-2026/'

# 1. Detailed results - most important for efficiency features
det_m = pd.read_csv(f'{DATA_DIR}/MRegularSeasonDetailedResults.csv')
print("=== MRegularSeasonDetailedResults ===")
print("Seasons:", det_m['Season'].min(), "–", det_m['Season'].max())
print("Rows:", len(det_m))
print("Columns:", list(det_m.columns))

# 2. Massey ordinals - key ranking feature
massey = pd.read_csv(f'{DATA_DIR}/MMasseyOrdinals.csv')
print("\n=== MMasseyOrdinals ===")
print("Seasons:", massey['Season'].min(), "–", massey['Season'].max())
print("Unique systems:", massey['SystemName'].nunique())
print("Sample systems:", massey['SystemName'].unique()[:15].tolist())
print("RankingDayNum values near tournament:", sorted(massey['RankingDayNum'].unique())[-5:])

# 3. Sample submission - what exactly we need to predict
sub2 = pd.read_csv(f'{DATA_DIR}/SampleSubmissionStage2.csv')
print("\n=== SampleSubmissionStage2 ===")
print("Rows:", len(sub2))
print("Seasons in submission:", sub2['ID'].str[:4].unique().tolist())
print("First 3 rows:", sub2.head(3).to_string())

=== MRegularSeasonDetailedResults ===
Seasons: 2003 – 2026
Rows: 124031
Columns: ['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF']

=== MMasseyOrdinals ===
Seasons: 2003 – 2026
Unique systems: 196
Sample systems: ['SEL', 'AP', 'BIH', 'DUN', 'ENT', 'GRN', 'IMS', 'MAS', 'MKV', 'MOR', 'POM', 'RPI', 'SAG', 'SAU', 'SE']
RankingDayNum values near tournament: [np.int64(125), np.int64(126), np.int64(127), np.int64(128), np.int64(133)]

=== SampleSubmissionStage2 ===
Rows: 132133
Seasons in submission: ['2026']
First 3 rows:                ID  Pred
0  2026_1101_1102   0.5
1  2026_1101_1103   0.5
2  2026_1101_1104   0.5


In [2]:
# Check seeds file
seeds = pd.read_csv(f'{DATA_DIR}/MNCAATourneySeeds.csv')
print("=== Seeds ===")
print("Seasons:", seeds['Season'].min(), "–", seeds['Season'].max())
print("Sample:", seeds[seeds['Season']==2025].head(5).to_string())

# Check tourney compact - our training labels
tourney = pd.read_csv(f'{DATA_DIR}/MNCAATourneyCompactResults.csv')
print("\n=== Tourney games per season (last 5) ===")
print(tourney.groupby('Season').size().tail(5))

# Women's detailed - does it exist and how many seasons?
det_w = pd.read_csv(f'{DATA_DIR}/WRegularSeasonDetailedResults.csv')
print("\n=== WRegularSeasonDetailedResults ===")
print("Seasons:", det_w['Season'].min(), "–", det_w['Season'].max())
print("Rows:", len(det_w))

# Massey - how many systems have DayNum=133 for 2025?
massey = pd.read_csv(f'{DATA_DIR}/MMasseyOrdinals.csv')
day133 = massey[(massey['Season']==2025) & (massey['RankingDayNum']==133)]
print("\n=== Massey systems at DayNum = 133 for 2025 ===")
print("Systems:", day133['SystemName'].nunique())
print("Teams ranked:", day133['TeamID'].nunique())

=== Seeds ===
Seasons: 1985 – 2025
Sample:       Season Seed  TeamID
2558    2025  W01    1181
2559    2025  W02    1104
2560    2025  W03    1458
2561    2025  W04    1112
2562    2025  W05    1332

=== Tourney games per season (last 5) ===
Season
2021    66
2022    67
2023    67
2024    67
2025    67
dtype: int64

=== WRegularSeasonDetailedResults ===
Seasons: 2010 – 2026
Rows: 86773

=== Massey systems at DayNum = 133 for 2025 ===
Systems: 56
Teams ranked: 364


In [3]:
# Seeds go back to 1985 — plenty of training data
# ~67 tourney games per season = solid training labels
# Women's detailed data from 2010 — enough for efficiency features
# 56 Massey systems at DayNum = 133 — perfect pre-tournament consensus signal

In [4]:
massey = pd.read_csv(f'{DATA_DIR}/MMasseyOrdinals.csv')

# Does 2026 have DayNum=133 rankings yet?
m2026 = massey[massey['Season'] == 2026]
print("=== 2026 Massey ===")
print("DayNums available:", sorted(m2026['RankingDayNum'].unique()))
print("Latest DayNum systems:", m2026[m2026['RankingDayNum'] == m2026['RankingDayNum'].max()]['SystemName'].nunique())

# Seeds for 2026 - do we have them yet?
seeds = pd.read_csv(f'{DATA_DIR}/MNCAATourneySeeds.csv')
print("\n=== 2026 Seeds ===")
print("2026 seeds available:", len(seeds[seeds['Season']==2026]))

# What's the latest regular season data for 2026?
reg = pd.read_csv(f'{DATA_DIR}/MRegularSeasonCompactResults.csv')
reg26 = reg[reg['Season']==2026]
print("\n=== 2026 Regular Season ===")
print("Games so far:", len(reg26))
print("Latest DayNum:", reg26['DayNum'].max())

=== 2026 Massey ===
DayNums available: [np.int64(2), np.int64(9), np.int64(16), np.int64(23), np.int64(30), np.int64(37), np.int64(44), np.int64(51), np.int64(58), np.int64(65), np.int64(72), np.int64(79), np.int64(86), np.int64(93), np.int64(100), np.int64(107), np.int64(114)]
Latest DayNum systems: 55

=== 2026 Seeds ===
2026 seeds available: 0

=== 2026 Regular Season ===
Games so far: 5149
Latest DayNum: 118


In [5]:
 # DayNum is a relative day counter within each season, not a calendar date.

In [6]:
# Elo built from 5,149 games so far
# Massey at DayNum = 114 (best available)
# Efficiency stats from this season
# No seeds yet — we'll use seed difference = 0 for 2026 (or skip it)

In [7]:
# Confirm women's tourney data
wt = pd.read_csv(f'{DATA_DIR}/WNCAATourneyCompactResults.csv')
print("=== Women's Tourney ===")
print("Seasons:", wt['Season'].min(), "–", wt['Season'].max())
print("Games per season (last 3):")
print(wt.groupby('Season').size().tail(3))

# Women's seeds
ws = pd.read_csv(f'{DATA_DIR}/WNCAATourneySeeds.csv')
print("\n2026 women's seeds:", len(ws[ws['Season']==2026]))

# Confirm submission is 2026 only
sub = pd.read_csv(f'{DATA_DIR}/SampleSubmissionStage2.csv')
ids = sub['ID'].str.split('_', expand=True)
print("\n=== Submission team ID ranges ===")
print("Min team1 ID:", ids[1].astype(int).min())
print("Max team1 ID:", ids[1].astype(int).max())
print("This tells us men's IDs (1000s) vs women's IDs (3000s) split:")
print(ids[1].astype(int).ge(3000).value_counts().rename({True:"Women",False:"Men"}))

=== Women's Tourney ===
Seasons: 1998 – 2025
Games per season (last 3):
Season
2023    67
2024    67
2025    67
dtype: int64

2026 women's seeds: 0

=== Submission team ID ranges ===
Min team1 ID: 1101
Max team1 ID: 3480
This tells us men's IDs (1000s) vs women's IDs (3000s) split:
1
Men      66430
Women    65703
Name: count, dtype: int64


## Key Findings
- **Men's detailed results:** 2003–2026 (124k games) → efficiency features available  
- **Women's detailed results:** 2010–2026 (87k games) → efficiency features available  
- **Massey ordinals:** 56 systems at DayNum=133 → strong pre-tournament consensus signal  
- **2026 status:** No seeds yet (Selection Sunday Mar 15) · Massey up to DayNum=114 · 5,149 regular season games played  
- **Submission:** 132,133 matchups — 66k men's + 66k women's, all 2026

## Feature Engineering
Three features per team, differenced for each matchup:

| Feature | Source | Notes |
|---|---|---|
| Elo rating | Compact results | MOV multiplier + home court adjustment |
| Net efficiency | Detailed results | (ORtg − DRtg) per 100 possessions |
| Massey median rank | MMasseyOrdinals | Median across 56 systems at latest DayNum |
| Seed | NCAATourneySeeds | 0 for 2026 until Selection Sunday |

In [8]:
# Load all CSVs 
m_reg     = pd.read_csv(f'{DATA_DIR}MRegularSeasonCompactResults.csv')
m_reg_det = pd.read_csv(f'{DATA_DIR}MRegularSeasonDetailedResults.csv')
m_tourney = pd.read_csv(f'{DATA_DIR}MNCAATourneyCompactResults.csv')
m_seeds   = pd.read_csv(f'{DATA_DIR}MNCAATourneySeeds.csv')

w_reg     = pd.read_csv(f'{DATA_DIR}WRegularSeasonCompactResults.csv')
w_reg_det = pd.read_csv(f'{DATA_DIR}WRegularSeasonDetailedResults.csv')
w_tourney = pd.read_csv(f'{DATA_DIR}WNCAATourneyCompactResults.csv')
w_seeds   = pd.read_csv(f'{DATA_DIR}WNCAATourneySeeds.csv')

massey    = pd.read_csv(f'{DATA_DIR}MMasseyOrdinals.csv')
sub       = pd.read_csv(f'{DATA_DIR}SampleSubmissionStage2.csv')

print("All CSVs loaded")
print(f"  Men's regular season games : {len(m_reg):,}")
print(f"  Women's regular season games: {len(w_reg):,}")

All CSVs loaded
  Men's regular season games : 198,079
  Women's regular season games: 142,093


# 2. Load Data

In [9]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ELO 
def compute_elo(reg_df, tourney_df, K=20, HCA=100, carry=0.75, init=1500):
    ratings, snapshot = {}, {}
    all_games = pd.concat([reg_df, tourney_df]).sort_values(['Season','DayNum'])
    current_season = None

    for _, row in all_games.iterrows():
        season = row['Season']
        if season != current_season:
            if current_season is not None:
                for tid, r in ratings.items():
                    snapshot[(current_season, tid)] = r
            ratings = {tid: init + carry*(r-init) for tid,r in ratings.items()}
            current_season = season

        w, l   = row['WTeamID'], row['LTeamID']
        rw, rl = ratings.get(w, init), ratings.get(l, init)
        loc    = row.get('WLoc','N')
        adj_rw = rw + (HCA if loc=='H' else -HCA if loc=='A' else 0)
        exp_w  = 1 / (1 + 10**((rl - adj_rw)/400))
        diff   = row['WScore'] - row['LScore']
        mov    = np.log(diff+1) * (2.2 / (abs(adj_rw-rl)*0.001 + 2.2))
        ratings[w] = rw + K*mov*(1-exp_w)
        ratings[l] = rl + K*mov*(0-(1-exp_w))

    for tid, r in ratings.items():
        snapshot[(current_season, tid)] = r
    return snapshot

# EFFICIENCY
def compute_efficiency(det_df):
    records = []
    for _, r in det_df.iterrows():
        def poss(fga, fta, orb, to):
            return max(fga - orb + to + 0.475*fta, 1)
        avg = (poss(r['WFGA'],r['WFTA'],r['WOR'],r['WTO']) +
               poss(r['LFGA'],r['LFTA'],r['LOR'],r['LTO'])) / 2
        for team, pts_f, pts_a in [(r['WTeamID'],r['WScore'],r['LScore']),
                                    (r['LTeamID'],r['LScore'],r['WScore'])]:
            records.append({'Season':r['Season'],'TeamID':team,
                            'PtsFor':pts_f,'PtsAgainst':pts_a,'Poss':avg})
    agg = pd.DataFrame(records).groupby(['Season','TeamID']).agg(
        TotFor=('PtsFor','sum'), TotAgainst=('PtsAgainst','sum'),
        TotPoss=('Poss','sum')).reset_index()
    agg['ORtg']   = 100 * agg['TotFor']     / agg['TotPoss']
    agg['DRtg']   = 100 * agg['TotAgainst'] / agg['TotPoss']
    agg['NetRtg'] = agg['ORtg'] - agg['DRtg']
    return agg.set_index(['Season','TeamID'])[['ORtg','DRtg','NetRtg']]

#  MASSEY MEDIAN
def compute_massey_median(massey_df):
    pre     = massey_df[massey_df['RankingDayNum'] <= 133]
    latest  = pre.groupby(['Season','SystemName'])['RankingDayNum'].max().reset_index()
    pre     = pre.merge(latest, on=['Season','SystemName','RankingDayNum'])
    return (pre.groupby(['Season','TeamID'])['OrdinalRank']
               .median().rename('MasseyMedian'))

# SEEDS 
def parse_seed(s): return int(s[1:3])

# RUN ALL 
print("Computing Men's Elo...")
m_elo = compute_elo(m_reg, m_tourney)

print("Computing Women's Elo...")
w_elo = compute_elo(w_reg, w_tourney)

print("Computing Men's efficiency...")
m_eff = compute_efficiency(m_reg_det)

print("Computing Women's efficiency...")
w_eff = compute_efficiency(w_reg_det)

print("Computing Massey median ranks...")
m_massey = compute_massey_median(massey)

m_seed_map = {(r.Season,r.TeamID): parse_seed(r.Seed) for _,r in m_seeds.iterrows()}
w_seed_map = {(r.Season,r.TeamID): parse_seed(r.Seed) for _,r in w_seeds.iterrows()}

print("\n All features ready")

Computing Men's Elo...
Computing Women's Elo...
Computing Men's efficiency...
Computing Women's efficiency...
Computing Massey median ranks...

 All features ready


# 3. Build Training Matrix
For each historical tournament game, we create one row with the feature differences between the two teams. Label = 1 if the lower TeamID won.

In [10]:
import xgboost as xgb
from sklearn.model_selection import cross_val_score
from sklearn.calibration import CalibratedClassifierCV

def build_features(season, t1, t2, elo_dict, eff_df, massey_series, seed_map, init=1500):
    """Return feature vector for a matchup. t1 < t2 always."""
    prev = season - 1

    # Elo
    e1 = elo_dict.get((season, t1), elo_dict.get((prev, t1), init))
    e2 = elo_dict.get((season, t2), elo_dict.get((prev, t2), init))

    # Efficiency
    def get_eff(s, tid):
        for ss in [s, s-1]:
            if (ss, tid) in eff_df.index:
                return eff_df.loc[(ss, tid)]
        return pd.Series({'ORtg':100,'DRtg':100,'NetRtg':0})

    eff1, eff2 = get_eff(season, t1), get_eff(season, t2)

    # Massey (men only — will be 0 for women)
    m1 = massey_series.get((season, t1), massey_series.get((prev, t1), 175))
    m2 = massey_series.get((season, t2), massey_series.get((prev, t2), 175))

    # Seed (0 if unknown)
    s1 = seed_map.get((season, t1), 8)
    s2 = seed_map.get((season, t2), 8)

    return [
        e1 - e2,                        # elo diff
        eff1['NetRtg'] - eff2['NetRtg'],# net efficiency diff
        eff1['ORtg']   - eff2['ORtg'],  # offensive rating diff
        eff1['DRtg']   - eff2['DRtg'],  # defensive rating diff (lower=better)
        m2 - m1,                        # massey rank diff (flipped: lower rank = better)
        s2 - s1,                        # seed diff (flipped: lower seed = better)
    ]

FEATURE_NAMES = ['elo_diff','net_rtg_diff','ortg_diff','drtg_diff','massey_diff','seed_diff']

def build_training_set(tourney_df, elo_dict, eff_df, massey_series, seed_map, min_season=2003):
    X, y = [], []
    for _, row in tourney_df.iterrows():
        season = row['Season']
        if season < min_season:
            continue
        w, l = row['WTeamID'], row['LTeamID']
        t1, t2 = (w,l) if w < l else (l,w)
        label  = 1 if w < l else 0

        feats = build_features(season, t1, t2, elo_dict, eff_df, massey_series, seed_map)
        X.append(feats)
        y.append(label)
    return np.array(X), np.array(y)

print("Building Men's training set...")
X_m, y_m = build_training_set(m_tourney, m_elo, m_eff, m_massey.to_dict(), m_seed_map)
print(f"  {len(y_m):,} games | features: {FEATURE_NAMES}")

print("Building Women's training set...")
w_massey_empty = {}  # no Massey for women
X_w, y_w = build_training_set(w_tourney, w_elo, w_eff, w_massey_empty, w_seed_map, min_season=2010)
print(f"  {len(y_w):,} games | features: {FEATURE_NAMES}")

# Combine men + women
X_all = np.vstack([X_m, X_w])
y_all = np.concatenate([y_m, y_w])
print(f"\nCombined: {len(y_all):,} total training games")

Building Men's training set...
  1,449 games | features: ['elo_diff', 'net_rtg_diff', 'ortg_diff', 'drtg_diff', 'massey_diff', 'seed_diff']
Building Women's training set...
  961 games | features: ['elo_diff', 'net_rtg_diff', 'ortg_diff', 'drtg_diff', 'massey_diff', 'seed_diff']

Combined: 2,410 total training games


# 4. Train XGBoost Model & Validate

In [11]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import brier_score_loss
import xgboost as xgb

# Train XGBoost 
model = xgb.XGBClassifier(
    n_estimators     = 200,
    max_depth        = 3,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    use_label_encoder= False,
    eval_metric      = 'logloss',
    random_state     = 42
)

# Cross-validate with Brier score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
brier_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_all, y_all)):
    X_tr, X_val = X_all[train_idx], X_all[val_idx]
    y_tr, y_val = y_all[train_idx], y_all[val_idx]

    model.fit(X_tr, y_tr)
    probs = model.predict_proba(X_val)[:,1]
    score = brier_score_loss(y_val, probs)
    brier_scores.append(score)
    print(f"  Fold {fold+1}: Brier = {score:.4f}")

print(f"\nMean CV Brier: {np.mean(brier_scores):.4f} ± {np.std(brier_scores):.4f}")
print(f"Baseline (0.5 always): 0.2500")
print(f"ADK starter benchmark: ~0.1670")

# Retrain on full data
model.fit(X_all, y_all)

# Feature importance 
print("\nFeature importances:")
for name, imp in sorted(zip(FEATURE_NAMES, model.feature_importances_),
                         key=lambda x: -x[1]):
    bar = "█" * int(imp * 50)
    print(f"  {name:<15} {imp:.3f}  {bar}")

  Fold 1: Brier = 0.1069
  Fold 2: Brier = 0.1180
  Fold 3: Brier = 0.1172
  Fold 4: Brier = 0.1188
  Fold 5: Brier = 0.1145

Mean CV Brier: 0.1151 ± 0.0043
Baseline (0.5 always): 0.2500
ADK starter benchmark: ~0.1670

Feature importances:
  elo_diff        0.465  ███████████████████████
  seed_diff       0.205  ██████████
  net_rtg_diff    0.124  ██████
  massey_diff     0.084  ████
  ortg_diff       0.062  ███
  drtg_diff       0.059  ██


# Results
#### - 0.1151 vs 0.1670 baseline — that's a 31% improvement.
## A few things stand out from the feature importances:

#### - Elo dominates at 46% — our MOV multiplier is doing real work
#### - Seed is strong at 20% — but we have zero 2026 seeds yet, so this will improve after Selection Sunday
#### - Net efficiency + Massey add meaningful signal on top

## 5. Separate Men's & Women's Models
Women's data has no Massey signal and different historical patterns.
Training them separately improves calibration for both.

In [12]:
# ── Recent form: win rate over last 14 days of regular season ─
def compute_recent_form(reg_df, window=14):
    """Win rate in last `window` days of regular season per team per season."""
    form = {}
    for season, grp in reg_df.groupby('Season'):
        max_day = grp['DayNum'].max()
        recent  = grp[grp['DayNum'] >= max_day - window]
        for _, row in recent.iterrows():
            w, l = row['WTeamID'], row['LTeamID']
            form[(season, w)] = form.get((season, w), []) + [1]
            form[(season, l)] = form.get((season, l), []) + [0]
    return {k: np.mean(v) for k, v in form.items()}

print("Computing recent form...")
m_form = compute_recent_form(m_reg)
w_form = compute_recent_form(w_reg)
print(f"  Men's form  : {len(m_form):,} entries")
print(f"  Women's form: {len(w_form):,} entries")

Computing recent form...
  Men's form  : 13,374 entries
  Women's form: 9,670 entries


In [13]:
# ── Tempo/pace feature ────────────────────────────────────────
def compute_tempo(det_df):
    """Average possessions per game per team per season."""
    records = []
    for _, r in det_df.iterrows():
        def poss(fga, fta, orb, to):
            return max(fga - orb + to + 0.475*fta, 1)
        wp = poss(r['WFGA'], r['WFTA'], r['WOR'], r['WTO'])
        lp = poss(r['LFGA'], r['LFTA'], r['LOR'], r['LTO'])
        avg = (wp + lp) / 2
        records.append({'Season':r['Season'],'TeamID':r['WTeamID'],'Poss':avg})
        records.append({'Season':r['Season'],'TeamID':r['LTeamID'],'Poss':avg})
    return pd.DataFrame(records).groupby(['Season','TeamID'])['Poss'].mean().rename('Tempo')

print("Computing tempo...")
m_tempo = compute_tempo(m_reg_det)
w_tempo = compute_tempo(w_reg_det)
print(f"  Men's tempo  : {len(m_tempo):,} entries")
print(f"  Women's tempo: {len(w_tempo):,} entries")

Computing tempo...
  Men's tempo  : 8,346 entries
  Women's tempo: 5,965 entries


In [14]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ── Helper ────────────────────────────────────────────────────
def get_eff(eff_df, s, tid):
    for ss in [s, s-1]:
        if (ss, tid) in eff_df.index:
            return eff_df.loc[(ss, tid), 'NetRtg']
    return 0.0

# ── Strength of schedule ──────────────────────────────────────
def compute_sos(reg_df, elo_dict):
    records = []
    for _, row in reg_df.iterrows():
        season = row['Season']
        w, l   = row['WTeamID'], row['LTeamID']
        records.append({'Season':season,'TeamID':w,'OppElo':elo_dict.get((season,l),1500)})
        records.append({'Season':season,'TeamID':l,'OppElo':elo_dict.get((season,w),1500)})
    return pd.DataFrame(records).groupby(['Season','TeamID'])['OppElo'].mean().rename('SOS')

print("Computing SOS...")
m_sos = compute_sos(m_reg, m_elo)
w_sos = compute_sos(w_reg, w_elo)
print(f"  Men's SOS:   {len(m_sos):,} entries")
print(f"  Women's SOS: {len(w_sos):,} entries")

# ── Feature names ─────────────────────────────────────────────
FEAT_NAMES = ['elo_diff','net_rtg_diff','massey_diff','seed_diff','sos_diff','form_diff','tempo_diff']

# ── Feature builder ───────────────────────────────────────────
def build_features_combined(season, t1, t2, is_women, init=1500):
    prev = season - 1
    elo  = w_elo  if is_women else m_elo
    eff  = w_eff  if is_women else m_eff
    smap = w_seed_map if is_women else m_seed_map
    sos  = w_sos  if is_women else m_sos
    frm  = w_form if is_women else m_form
    tmp  = w_tempo if is_women else m_tempo

    e1 = elo.get((season,t1), elo.get((prev,t1), init))
    e2 = elo.get((season,t2), elo.get((prev,t2), init))
    n1 = get_eff(eff, season, t1)
    n2 = get_eff(eff, season, t2)
    s1 = smap.get((season,t1), 8)
    s2 = smap.get((season,t2), 8)
    o1 = sos.to_dict().get((season,t1), sos.to_dict().get((prev,t1), 1500))
    o2 = sos.to_dict().get((season,t2), sos.to_dict().get((prev,t2), 1500))
    f1 = frm.get((season,t1), frm.get((prev,t1), 0.5))
    f2 = frm.get((season,t2), frm.get((prev,t2), 0.5))
    tp1 = tmp.to_dict().get((season,t1), tmp.to_dict().get((prev,t1), 70.0))
    tp2 = tmp.to_dict().get((season,t2), tmp.to_dict().get((prev,t2), 70.0))

    if not is_women:
        mm = m_massey.to_dict()
        m1 = mm.get((season,t1), mm.get((prev,t1), 175))
        m2 = mm.get((season,t2), mm.get((prev,t2), 175))
    else:
        m1, m2 = 175, 175

    return [e1-e2, n1-n2, m2-m1, s2-s1, o1-o2, f1-f2, tp1-tp2]

# ── Training set builder ──────────────────────────────────────
def build_training_combined(min_season_m=2003, min_season_w=2010):
    X, y = [], []
    for _, row in m_tourney.iterrows():
        if row['Season'] < min_season_m: continue
        w, l   = row['WTeamID'], row['LTeamID']
        t1, t2 = (w,l) if w<l else (l,w)
        X.append(build_features_combined(row['Season'], t1, t2, False))
        y.append(1 if w<l else 0)
    for _, row in w_tourney.iterrows():
        if row['Season'] < min_season_w: continue
        w, l   = row['WTeamID'], row['LTeamID']
        t1, t2 = (w,l) if w<l else (l,w)
        X.append(build_features_combined(row['Season'], t1, t2, True))
        y.append(1 if w<l else 0)
    return np.array(X), np.array(y)

# ── Build training set ────────────────────────────────────────
print("\nBuilding combined training set...")
X_comb, y_comb = build_training_combined()
print(f"Total samples: {len(y_comb):,}")

# ── Tune XGBoost ──────────────────────────────────────────────
configs = [
    dict(n_estimators=100, max_depth=2, learning_rate=0.05, min_child_weight=5),
    dict(n_estimators=200, max_depth=2, learning_rate=0.05, min_child_weight=5),
    dict(n_estimators=100, max_depth=3, learning_rate=0.05, min_child_weight=5),
    dict(n_estimators=200, max_depth=3, learning_rate=0.05, min_child_weight=7),
    dict(n_estimators=300, max_depth=2, learning_rate=0.02, min_child_weight=5),
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("\nTuning XGBoost configs:")
best_score, best_cfg = 999, None

for cfg in configs:
    m = xgb.XGBClassifier(**cfg, subsample=0.8, colsample_bytree=0.8,
                           eval_metric='logloss', random_state=42)
    scores = []
    for tr, val in cv.split(X_comb, y_comb):
        m.fit(X_comb[tr], y_comb[tr])
        scores.append(brier_score_loss(y_comb[val], m.predict_proba(X_comb[val])[:,1]))
    mean_s = np.mean(scores)
    marker = " ◄ best" if mean_s < best_score else ""
    print(f"  depth={cfg['max_depth']} trees={cfg['n_estimators']:3d} "
          f"lr={cfg['learning_rate']} mcw={cfg['min_child_weight']}  "
          f"Brier={mean_s:.4f}{marker}")
    if mean_s < best_score:
        best_score, best_cfg = mean_s, cfg

print(f"\nBest CV Brier: {best_score:.4f}")
print(f"Best config  : {best_cfg}")

# ── Train final XGBoost ───────────────────────────────────────
model_final = xgb.XGBClassifier(**best_cfg, subsample=0.8, colsample_bytree=0.8,
                                  eval_metric='logloss', random_state=42)
model_final.fit(X_comb, y_comb)

print("\nFeature importances:")
for name, imp in sorted(zip(FEAT_NAMES, model_final.feature_importances_),
                         key=lambda x: -x[1]):
    print(f"  {name:<15} {imp:.3f}  {'█'*int(imp*50)}")

# ── Ensemble: XGBoost + Logistic Regression ───────────────────
print("\nTesting ensemble...")
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_comb)
blend_scores = []

for tr, val in cv.split(X_comb, y_comb):
    xgb_m = xgb.XGBClassifier(**best_cfg, subsample=0.8,
                                colsample_bytree=0.8,
                                eval_metric='logloss', random_state=42)
    xgb_m.fit(X_comb[tr], y_comb[tr])
    xgb_preds = xgb_m.predict_proba(X_comb[val])[:,1]

    lr_m = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    lr_m.fit(scaler.transform(X_comb[tr]), y_comb[tr])
    lr_preds = lr_m.predict_proba(scaler.transform(X_comb[val]))[:,1]

    blended = 0.70 * xgb_preds + 0.30 * lr_preds
    blend_scores.append(brier_score_loss(y_comb[val], blended))

print(f"XGBoost alone  : {best_score:.4f}")
print(f"Ensemble 70/30 : {np.mean(blend_scores):.4f} ± {np.std(blend_scores):.4f}")

if np.mean(blend_scores) < best_score:
    print("\n Ensemble is better — using blended model")
    lr_final = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    lr_final.fit(X_scaled, y_comb)
    USE_ENSEMBLE = True
else:
    print("\n Ensemble not better — keeping XGBoost only")
    USE_ENSEMBLE = False

Computing SOS...
  Men's SOS:   13,753 entries
  Women's SOS: 9,851 entries

Building combined training set...
Total samples: 2,410

Tuning XGBoost configs:
  depth=2 trees=100 lr=0.05 mcw=5  Brier=0.1182 ◄ best
  depth=2 trees=200 lr=0.05 mcw=5  Brier=0.1108 ◄ best
  depth=3 trees=100 lr=0.05 mcw=5  Brier=0.1117
  depth=3 trees=200 lr=0.05 mcw=7  Brier=0.1084 ◄ best
  depth=2 trees=300 lr=0.02 mcw=5  Brier=0.1164

Best CV Brier: 0.1084
Best config  : {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'min_child_weight': 7}

Feature importances:
  elo_diff        0.401  ████████████████████
  seed_diff       0.215  ██████████
  net_rtg_diff    0.104  █████
  sos_diff        0.094  ████
  massey_diff     0.075  ███
  form_diff       0.067  ███
  tempo_diff      0.044  ██

Testing ensemble...
XGBoost alone  : 0.1084
Ensemble 70/30 : 0.1052 ± 0.0028

 Ensemble is better — using blended model


In [15]:
# Check if 2026 seeds have been released yet
# seeds_2026 = pd.read_csv(f'{DATA_DIR}MNCAATourneySeeds.csv')
# print("2026 men's seeds:", len(seeds_2026[seeds_2026['Season']==2026]))

# w_seeds_2026 = pd.read_csv(f'{DATA_DIR}WNCAATourneySeeds.csv')
# print("2026 women's seeds:", len(w_seeds_2026[w_seeds_2026['Season']==2026]))

In [16]:
# ── Fast vectorized submission with ensemble ──────────────────
print("Generating predictions...")

parts    = sub['ID'].str.split('_', expand=True)
seasons  = parts[0].astype(int).values
t1s      = parts[1].astype(int).values
t2s      = parts[2].astype(int).values
is_women = t1s >= 3000

all_feats = np.array([
    build_features_combined(seasons[i], t1s[i], t2s[i], is_women[i])
    for i in range(len(seasons))
])

# Ensemble prediction
xgb_preds = model_final.predict_proba(all_feats)[:,1]
lr_preds  = lr_final.predict_proba(scaler.transform(all_feats))[:,1]

if USE_ENSEMBLE:
    preds = 0.70 * xgb_preds + 0.30 * lr_preds
else:
    preds = xgb_preds

preds = np.clip(preds, 0.025, 0.975)
sub['Pred'] = preds
sub.to_csv('/kaggle/working/submission.csv', index=False)

print(f"✅ Done!")
print(f"Predictions : {len(preds):,}")
print(f"Mean        : {np.mean(preds):.4f}")
print(f"Std         : {np.std(preds):.4f}")
print(f"Min / Max   : {np.min(preds):.4f} / {np.max(preds):.4f}")

Generating predictions...
✅ Done!
Predictions : 132,133
Mean        : 0.4930
Std         : 0.3990
Min / Max   : 0.0250 / 0.9750
